# SMS Spam Classification - Model Training & Saving

Train benchmark models from previous assignments and save the best model as a sklearn Pipeline (vectorizer + classifier) using joblib for use in scoring and serving.

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, auc
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
# load train, validation, and test splits from previous assignments
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

X_train, y_train = train_df['message'], train_df['label']
X_val, y_val = val_df['message'], val_df['label']
X_test, y_test = test_df['message'], test_df['label']

print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

Train: 3900, Validation: 836, Test: 836


## Train 3 benchmark models as Pipelines

Using sklearn Pipeline so the vectorizer is bundled with the classifier.
This way the saved model can directly accept raw text input.

In [3]:
# define 3 pipelines (same models as Assign1 and Assign2)
pipelines = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
        ('clf', LogisticRegression(max_iter=1000, C=10))
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
        ('clf', MultinomialNB(alpha=1.0))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
        ('clf', LinearSVC(max_iter=1000))
    ])
}

# train all models
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    print(f"Trained {name}")

Trained Logistic Regression
Trained Naive Bayes
Trained Linear SVM


In [4]:
# evaluate all models on test set
for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"{name} -- Accuracy: {acc:.4f}, F1: {f1:.4f}")

Logistic Regression -- Accuracy: 0.9737, F1: 0.8922
Naive Bayes -- Accuracy: 0.9725, F1: 0.8856
Linear SVM -- Accuracy: 0.9785, F1: 0.9135


## Save the best model

LinearSVC was the best in previous assignments, but it does not support predict_proba
which we need for the propensity score in score.py.
So we pick the best model that supports predict_proba -- Logistic Regression (with C=10 from tuning in Assign1).

In [5]:
best_model = pipelines['Logistic Regression']

# save the pipeline (vectorizer + classifier) to a file
joblib.dump(best_model, 'best_model.pkl')
print("Saved best model to best_model.pkl")

Saved best model to best_model.pkl


In [6]:
# quick test: load it back and make sure it works on raw text
loaded = joblib.load('best_model.pkl')
test_text = "Congratulations! You won a free prize"
proba = loaded.predict_proba([test_text])[0]
print(f"Test prediction on: '{test_text}'")
print(f"P(ham)={proba[0]:.4f}, P(spam)={proba[1]:.4f}")

Test prediction on: 'Congratulations! You won a free prize'
P(ham)=0.0708, P(spam)=0.9292
